# Query rewriting and expansion: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2 langchain-core==1.8.4

## Why rewrite a query before searching

MiniLM ranks for four typed questions and their rewrites.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
pairs = [
    ("leave-parental", "kid on the way, what do I get",
     "paid parental leave for a new child"),
    ("exp-approval", "got a pricey monitor, who has to sign off",
     "approval needed for a large expense claim"),
    ("leave-carry", "holidays I did not use, do they vanish",
     "carry over unused annual leave"),
    ("run-sev", "Sev 2 response time",
     "how quickly does support reply"),
]

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
passages = model.encode([docs[i] for i in ids], normalize_embeddings=True)
for right, typed, rewritten in pairs:
    for label, text in (("typed    ", typed), ("rewritten", rewritten)):
        sims = passages @ model.encode(text, normalize_embeddings=True)
        order = sorted(range(len(ids)), key=lambda j: (-float(sims[j]), ids[j]))
        rank = [ids[j] for j in order].index(right) + 1
        print(f"{label} rank {rank:2d}  cosine {float(sims[ids.index(right)]):.3f}  {text}")

typed     rank  2  cosine 0.098  kid on the way, what do I get
rewritten rank  1  cosine 0.588  paid parental leave for a new child
typed     rank  4  cosine 0.158  got a pricey monitor, who has to sign off
rewritten rank  1  cosine 0.603  approval needed for a large expense claim
typed     rank  2  cosine 0.459  holidays I did not use, do they vanish
rewritten rank  1  cosine 0.801  carry over unused annual leave
typed     rank  1  cosine 0.757  Sev 2 response time
rewritten rank  1  cosine 0.561  how quickly does support reply


## HyDE: search with a hypothetical passage

all-MiniLM-L6-v2 ranks: question, hypothetical passage, and the average of both vectors.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
cases = [
    ("v02", "leave-parental",
     "kid on the way, what do I get",
     "Employees who are having a child receive several weeks of paid parental leave."),
    ("v08", "exp-approval",
     "got a pricey monitor, who has to sign off",
     "Large expense claims need a receipt and approval from a manager, and very large ones also from finance."),
    ("q01", "kb-err5103",
     "ERR-5103",
     "A problem happened while importing data."),
    ("q02", "kb-err5031",
     "ERR-5031",
     "A problem happened while exporting data."),
    ("q07", "exp-meals",
     "Tier 2 dinner limit",
     "Employees can claim meals during business travel."),
    ("q08", "leave-carry",
     "31 March",
     "Some things must be finished before a deadline."),
]

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
passages = model.encode([docs[i] for i in ids], normalize_embeddings=True)

def rank(vector, right):
    sims = passages @ vector
    order = sorted(range(len(ids)), key=lambda j: (-float(sims[j]), ids[j]))
    return [ids[j] for j in order].index(right) + 1

for qid, right, question, fake in cases:
    q, h = model.encode([question, fake], normalize_embeddings=True)
    print(f"{qid}: question {rank(q, right)}  hypothetical {rank(h, right)}  average of both {rank((q + h) / 2, right)}")

v02: question 2  hypothetical 1  average of both 1
v08: question 4  hypothetical 1  average of both 1
q01: question 4  hypothetical 1  average of both 1
q02: question 3  hypothetical 2  average of both 2
q07: question 1  hypothetical 3  average of both 1
q08: question 1  hypothetical 3  average of both 1


## Multi-query retrieval and fusion

A variants chain with a fake model, then a real search and fusion.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
question = "what do we owe them if the service was flaky this month"
REPLY = "1. uptime commitment and service credit\n2. refund when availability is below the promise\n\n3. what happens if uptime is missed\n4. uptime commitment and service credit"     # hand-written: what a model might answer

import re
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from sentence_transformers import SentenceTransformer

def parse_variants(reply):
    lines = [re.sub(r"^\s*\d+[.)]\s*", "", line).strip() for line in reply.splitlines()]
    return list(dict.fromkeys(line for line in lines if line))[:3]     # no blanks, no repeats

prompt = ChatPromptTemplate.from_messages([
    ("system", "Write {n} different search queries for the user's question, one per line. "
               "Keep every code and number. Do not answer the question."),
    ("human", "{question}")])
fake_model = FakeListChatModel(responses=[REPLY])          # FAKE: always returns REPLY
chain = prompt | fake_model | StrOutputParser() | RunnableLambda(parse_variants)
variants = chain.invoke({"n": 3, "question": question})
print("variants:", len(variants))

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)

def search(text):
    cosines = vectors @ model.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]
    return [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))]

score = {}
for text in [question] + variants:
    ranking = search(text)
    print(f"  faq-uptime at rank {ranking.index('faq-uptime') + 1}: {text}")
    for rank, pid in enumerate(ranking, 1):
        score[pid] = score.get(pid, 0.0) + 1.0 / (60 + rank)
fused = sorted(score, key=lambda p: (-score[p], p))
print("fused top 3:", fused[:3])

variants: 3
  faq-uptime at rank 2: what do we owe them if the service was flaky this month
  faq-uptime at rank 1: uptime commitment and service credit
  faq-uptime at rank 4: refund when availability is below the promise
  faq-uptime at rank 3: what happens if uptime is missed
fused top 3: ['run-sev', 'faq-uptime', 'faq-trash']


## Step-back questions and decomposition

One search against two, on all 28 passages.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
cases = {  # (whole question, two hand-written sub-questions, the two right passages)
    "c01": ("I am flying to a client and will pay 8000 rupees for the ticket: which class can I book and who approves the claim?",
            ["which flight class can I book", "who approves a claim of 8000 rupees"],
            ["exp-travel", "exp-approval"]),
    "c06": ("Right after a release the service is down for most customers: how fast must we respond and what do we run first?",
            ["response time when the service is down for most customers", "what to run first when a release causes a problem"],
            ["run-sev", "run-rollback"]),
}
from itertools import zip_longest
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)

def search(text):
    cosines = vectors @ model.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]
    return [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))]

for cid, (question, parts, right) in cases.items():
    one = search(question)[:3]
    merged = [p for pair in zip_longest(*[search(part) for part in parts]) for p in pair]
    split = list(dict.fromkeys(merged))[:3]
    print(cid, "ranks of the right passages in one search:", [search(question).index(p) + 1 for p in right])
    print("   one search:", one)
    print("   split     :", split)

c01 ranks of the right passages in one search: [2, 1]
   one search: ['exp-approval', 'exp-travel', 'kb-merge']
   split     : ['exp-travel', 'exp-approval', 'faq-seats']
c06 ranks of the right passages in one search: [1, 7]
   one search: ['run-sev', 'faq-uptime', 'sec-devices']
   split     : ['run-sev', 'run-rollback', 'faq-uptime']


## Using the conversation history in retrieval

A condense chain with a scripted stand-in model, then a real embedding search.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.language_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import SentenceTransformer

docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
chats = {
    "h2": ([('user', 'What does error ERR-5031 mean?'), ('assistant', 'The export file is larger than 2 GB.')],
           'and the next number after it?', 'kb-err5032'),
    "h5": ([('user', 'Can I book business class to visit a client?'), ('assistant', 'Only with written approval for short flights.')],
           'and who gives it?', 'exp-travel'),
}
prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the last message into one search query. Do not answer it."),
    MessagesPlaceholder("history"), ("human", "{question}")])
# FAKE model: it returns these hand-written replies in order and reads nothing
model = FakeListChatModel(responses=['what does error ERR-5032 mean?',
                                     'who gives written approval to book business class?'])
condense = prompt | model | StrOutputParser()
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
matrix = embedder.encode([docs[i] for i in ids], normalize_embeddings=True)
def rank_of(query, right):
    scores = matrix @ embedder.encode(query, normalize_embeddings=True)
    order = sorted(range(len(ids)), key=lambda i: -scores[i])
    return [ids[i] for i in order].index(right) + 1

for name, (history, last, right) in chats.items():
    messages = prompt.invoke({"history": history, "question": last}).to_messages()
    query = condense.invoke({"history": history, "question": last})
    print(name, "messages sent to the model:", len(messages))
    print("   searched:", query)
    print("   rank as typed:", rank_of(last, right), "| rank condensed:", rank_of(query, right))

h2 messages sent to the model: 4
   searched: what does error ERR-5032 mean?
   rank as typed: 25 | rank condensed: 1
h5 messages sent to the model: 4
   searched: who gives written approval to book business class?
   rank as typed: 18 | rank condensed: 1


## Routing, and when not to rewrite

Always against routed rewriting on the exact questions, with the real model.

In [ ]:
from sentence_transformers import SentenceTransformer
import re

def route(question):
    # 'keep': search with the words as typed (a number, a code or a one or two word question). 'rewrite': rewrite first.
    if re.search(r"[0-9]", question) or len(question.split()) <= 2:
        return "keep"
    return "rewrite"

docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
# the eight exact-word questions with their hand-written rewrites
cases = [
    ('ERR-5103', 'an import fails with a file error', 'kb-err5103'),
    ('ERR-5031', 'an export fails with a file error', 'kb-err5031'),
    ('ERR-4012', 'a sign-in error occurred', 'kb-err4012'),
    ('Sev 2 response time', 'how quickly does support reply', 'run-sev'),
    ('Frankfurt', 'data location in Europe', 'faq-region'),
    ('10,000 tickets', 'too many tickets', 'kb-slow'),
    ('Tier 2 dinner limit', 'meal allowance while travelling', 'exp-meals'),
    ('31 March', 'end of year deadline', 'leave-carry'),
]

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
matrix = model.encode([docs[i] for i in ids], normalize_embeddings=True)
def rank_of(query, right):
    scores = matrix @ model.encode(query, normalize_embeddings=True)
    order = sorted(range(len(ids)), key=lambda i: -scores[i])
    return [ids[i] for i in order].index(right) + 1

always, routed = [], []
for typed, rewritten, right in cases:
    a, b = rank_of(typed, right), rank_of(rewritten, right)
    always.append(b)
    routed.append(b if route(typed) == "rewrite" else a)
    print(f"{typed:20s} typed {a}  rewritten {b}  rule: {route(typed)}")
for name, ranks in (("always", always), ("routed", routed)):
    print(name, "rank 1 for", sum(r == 1 for r in ranks), "of", len(ranks))

ERR-5103             typed 4  rewritten 1  rule: keep
ERR-5031             typed 3  rewritten 2  rule: keep
ERR-4012             typed 3  rewritten 3  rule: keep
Sev 2 response time  typed 1  rewritten 1  rule: keep
Frankfurt            typed 1  rewritten 1  rule: keep
10,000 tickets       typed 2  rewritten 2  rule: keep
Tier 2 dinner limit  typed 1  rewritten 1  rule: keep
31 March             typed 1  rewritten 1  rule: keep
always rank 1 for 5 of 8
routed rank 1 for 4 of 8


## Notebook: a query-transform layer end to end

Typed against routed, for everyday and exact-word questions.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
# (typed question, hand-written rewrite, right passage); 'e' = everyday words, 'x' = exact words
single = [
    ('e', 'someone nicked my work laptop', 'report a stolen company laptop to the security team', 'sec-devices'),
    ('e', 'kid on the way, what do I get', 'paid parental leave for a new child', 'leave-parental'),
    ('e', 'it keeps bouncing me back to sign in', 'sign-in page keeps reloading', 'kb-login-loop'),
    ('e', 'we are eight people, is the cheapest option enough', 'how many seats does the Starter plan include', 'faq-seats'),
    ('e', 'what do we owe them if the service was flaky this month', 'credit when monthly uptime falls below the commitment', 'faq-uptime'),
    ('e', 'binned a ticket by mistake, is it gone', 'how long do deleted tickets stay in the trash', 'faq-trash'),
    ('e', 'do I need a note from the doc when I am ill', 'when is a medical certificate required for sick leave', 'leave-sick'),
    ('e', 'got a pricey monitor, who has to sign off', 'approval needed for a large expense claim', 'exp-approval'),
    ('e', 'the whole thing is down, how fast must I jump', 'response time for a Sev 1 outage', 'run-sev'),
    ('e', 'client says they wrote to us but nothing arrived', 'customer email missing from the inbox', 'kb-mail'),
    ('e', 'everything broke right after we shipped', 'roll back a release that caused a problem', 'run-rollback'),
    ('e', 'where does our info physically live', 'which region is customer data stored in', 'faq-region'),
    ('e', 'my six digit number gets rejected', 'authenticator code rejected at sign-in', 'kb-err4021'),
    ('e', 'fat fingered it too many times and now I am stuck', 'account locked after wrong password attempts', 'kb-err4013'),
    ('e', 'same person opened two threads, can I squash them', 'merge two tickets from the same customer', 'kb-merge'),
    ('e', 'holidays I did not use, do they vanish', 'carry over unused annual leave', 'leave-carry'),
    ('x', 'ERR-5103', 'an import fails with a file error', 'kb-err5103'),
    ('x', 'ERR-5031', 'an export fails with a file error', 'kb-err5031'),
    ('x', 'ERR-4012', 'a sign-in error occurred', 'kb-err4012'),
    ('x', 'Sev 2 response time', 'how quickly does support reply', 'run-sev'),
    ('x', 'Frankfurt', 'data location in Europe', 'faq-region'),
    ('x', '10,000 tickets', 'too many tickets', 'kb-slow'),
    ('x', 'Tier 2 dinner limit', 'meal allowance while travelling', 'exp-meals'),
    ('x', '31 March', 'end of year deadline', 'leave-carry'),
]
import re

def route(question):
    # 'keep': search with the words as typed (a number, a code or a one or two word question). 'rewrite': rewrite first.
    if re.search(r"[0-9]", question) or len(question.split()) <= 2:
        return "keep"
    return "rewrite"

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
matrix = model.encode([docs[i] for i in ids], normalize_embeddings=True)

def search(query):
    scores = matrix @ model.encode(query, normalize_embeddings=True)
    return [ids[i] for i in sorted(range(len(ids)), key=lambda i: (-scores[i], ids[i]))]

for group, name in (("e", "everyday"), ("x", "exact")):
    before, after = [], []
    for g, typed, rewritten, right in single:
        if g != group:
            continue
        query = rewritten if route(typed) == "rewrite" else typed
        before.append(search(typed).index(right) + 1)
        after.append(search(query).index(right) + 1)
    n = len(before)
    for label, ranks in (("typed ", before), ("routed", after)):
        print(f"{name:8s} {label} recall@1 {sum(r == 1 for r in ranks) / n:.4f}  recall@3 {sum(r <= 3 for r in ranks) / n:.4f}")

everyday typed  recall@1 0.6250  recall@3 0.9375
everyday routed recall@1 1.0000  recall@3 1.0000
exact    typed  recall@1 0.5000  recall@3 0.8750
exact    routed recall@1 0.5000  recall@3 0.8750


Split two-part questions and condense follow-ups.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
# compound: (question, [sub-questions], [right passages])
compound = [
    ('I am flying to a client and will pay 8000 rupees for the ticket: which class can I book and who approves the claim?', ['which flight class can I book', 'who approves a claim of 8000 rupees'], ['exp-travel', 'exp-approval']),
    ('A laptop was stolen and it had customer data in my personal notes: how fast must I report it, and was keeping that data there allowed?', ['how fast must a stolen laptop be reported', 'may customer data be copied into personal notes'], ['sec-devices', 'sec-data']),
    ('I will be sick for four days and later want annual leave: what proof do I need and how early must I ask for the leave?', ['what proof is needed for four sick days', 'how early must annual leave be requested'], ['leave-sick', 'leave-apply']),
    ('Which plan do we need for single sign-on, and what uptime does that plan promise?', ['which plan includes single sign-on', 'what uptime does the Business plan promise'], ['faq-sso', 'faq-uptime']),
    ("A customer's export link stopped working and their next export is bigger than 2 GB: what should they do about each?", ['the export link has expired', 'the export file is larger than 2 GB'], ['kb-err5032', 'kb-err5031']),
    ('Right after a release the service is down for most customers: how fast must we respond and what do we run first?', ['response time when the service is down for most customers', 'what to run first when a release causes a problem'], ['run-sev', 'run-rollback']),
]
# conversations: (last message, stand-alone question, right passage)
chats = [
    ('can I move some of them to next year?', 'can I carry unused annual leave days over to next year?', 'leave-carry'),
    ('and the next number after it?', 'what does error ERR-5032 mean?', 'kb-err5032'),
    ('what does it promise about downtime?', 'what uptime does the Business plan promise?', 'faq-uptime'),
    ('how long do I have for that?', 'how fast must a stolen laptop be reported to the security team?', 'sec-devices'),
    ('and who gives it?', 'who gives written approval to book business class?', 'exp-travel'),
    ('Different thing: how do I join two tickets?', 'how do I merge two tickets?', 'kb-merge'),
    ('ok. How much leave does the second parent get?', 'how long is parental leave for the second caregiver?', 'leave-parental'),
    ('can we pick somewhere else?', 'can we choose another region for customer data?', 'faq-region'),
]
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
ids = list(docs)
matrix = model.encode([docs[i] for i in ids], normalize_embeddings=True)

def search(query):
    scores = matrix @ model.encode(query, normalize_embeddings=True)
    return [ids[i] for i in sorted(range(len(ids)), key=lambda i: (-scores[i], ids[i]))]

def merge(lists, k=3):
    out = []
    for depth in range(len(ids)):
        for x in lists:
            if x[depth] not in out:
                out.append(x[depth])
    return out[:k]

one = sum(len(set(search(q)[:3]) & set(right)) / 2 for q, subs, right in compound) / len(compound)
split = sum(len(set(merge([search(s) for s in subs])) & set(right)) / 2 for q, subs, right in compound) / len(compound)
print(f"compound, share of right passages in top 3: one search {one:.4f}, split {split:.4f}")
for label, position in (("last message", 0), ("stand-alone ", 1)):
    ranks = [search(c[position]).index(c[2]) + 1 for c in chats]
    print(f"chats, {label} recall@1 {sum(r == 1 for r in ranks) / len(ranks):.4f}  ranks {ranks}")

compound, share of right passages in top 3: one search 0.9167, split 1.0000
chats, last message recall@1 0.5000  ranks [1, 25, 2, 4, 18, 1, 1, 1]
chats, stand-alone  recall@1 1.0000  ranks [1, 1, 1, 1, 1, 1, 1, 1]
